# 02 · Modelos, calibración y degradación temporal

Qué se responde aquí:

1. ¿Cuál de las tres familias decide mejor, y según qué criterio?
2. ¿Sirve de algo calibrar?
3. ¿Se degrada el sistema estático, y cómo se distingue eso de un cambio de tasa base?

Lee artefactos ya calculados: **no reentrena**.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "configs").exists() and (ROOT.parent / "configs").exists():
    ROOT = ROOT.parent          # el notebook puede abrirse desde notebooks/
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fraud_adaptive.pipeline import load_configs
from fraud_adaptive.tracking import read_json

CONFIGS = load_configs(ROOT / "configs")
RUN_DIR = ROOT / CONFIGS["base"]["paths"]["runs"] / "principal"
REPORTS = ROOT / CONFIGS["base"]["paths"]["reports"]

MANIFEST = read_json(ROOT / CONFIGS["base"]["paths"]["manifests"] / "sources.json")
DATA_SOURCE = MANIFEST.get("data_source", "desconocido")

if DATA_SOURCE == "sintetico_sustituto":
    print("AVISO: los datos son un SUSTITUTO SINTETICO. Ninguna cifra describe IEEE-CIS.")
else:
    print("Datos: IEEE-CIS Fraud Detection (particion train).")
print("Run:", RUN_DIR)

## 1. Tuning: 18 fits

3 familias × 3 configuraciones × 2 folds forward. La selección es por AP media de validación; los folds son **internos** y sus métricas nunca se reportan como resultado final.

In [ ]:
tuning = pd.read_csv(REPORTS / "tables" / "tuning.csv")
tuning.sort_values(["familia", "ap_media"], ascending=[True, False]).round(4)

## 2. Comparación de las tres familias

El criterio de decisión es el **costo**, no el AP. Una familia con mejor AP puede decidir peor si su calibración desplaza los umbrales.

In [ ]:
modelos = pd.read_csv(REPORTS / "tables" / "comparacion_modelos.csv")
modelos[["familia", "config", "ap_politica", "costo_um_tx_politica",
         "brier_crudo", "brier_calibrado", "ece_crudo", "ece_calibrado",
         "tau_low", "tau_high", "elegida_para_adaptacion"]].round(5)

### Por qué calibrar

La política compara `p·monto` con `(1−p)·c_FP`. Esa aritmética solo tiene sentido si
`p` es una probabilidad, no un puntaje ordenado. Un modelo con `scale_pos_weight`
produce puntajes inflados: usarlos como probabilidad haría bloquear de más, y el
AP no lo notaría.

In [ ]:
estaticos = read_json(RUN_DIR / "static_results.json")

for familia, info in estaticos["familias"].items():
    cal = info["calibracion"]
    print(f"{familia:<15} Brier {cal['brier_crudo']:.5f} -> {cal['brier_calibrado']:.5f}"
          f"   ECE {cal['ece_crudo']:.5f} -> {cal['ece_calibrado']:.5f}")

print(f"\nFamilia congelada para adaptación: {estaticos['familia_elegida']}")
print(f"Hash de prerregistro: {estaticos['prerregistro']['hash'][:32]}")

## 3. Umbrales: nunca 0.5

Se eligen por costo sobre la reserva `[76,83)`. También se fijan dos umbrales **diagnósticos**, que se aplican congelados al test.

In [ ]:
elegida = estaticos["familias"][estaticos["familia_elegida"]]

print("Política económica:", json.dumps(elegida["policy"], indent=2))
print("\nUmbral para FPR<=1%   :", json.dumps(elegida["umbral_fpr"], indent=2))
print("Umbral para precisión>=80%:", json.dumps(elegida["umbral_precision"], indent=2))
print("\nReferencias simuladas (UM/tx):")
for nombre, valores in estaticos["baselines"].items():
    print(f"   {nombre:<18} {valores['costo_por_tx']:.4f}")

## 4. Degradación del sistema estático

La evidencia clave no es que el AP baje, sino que baje **mientras la tasa base se
mantiene plana**. Si la prevalencia cayera a la vez, el AP bajaría por aritmética y
no habría nada que concluir sobre el modelo.

In [ ]:
from fraud_adaptive.metrics import metrics_by_period

desenlaces = pd.read_parquet(RUN_DIR / "outcomes.parquet")
estatico = desenlaces[desenlaces["estrategia"] == "S0"]

serie = metrics_by_period(estatico, period_column="semana",
                          threshold_binary=elegida["policy"]["tau_high"])
serie[["semana", "n", "prevalencia", "ap", "skill", "brier", "costo_por_tx"]].round(4)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(REPORTS / "figures" / "rendimiento_estatico.png")))
display(Image(filename=str(REPORTS / "figures" / "calibracion_costos.png")))

## 5. Lectura

Si el AP cae y la prevalencia no, el modelo perdió capacidad discriminante sobre
las mismas clases: es consistente con un cambio en P(y|X). La comparación temporal
por sí sola **no identifica causalmente** el drift, y el informe no lo afirma.